# TARDIS
This notebook loads the dataset, cleans the data, handles missing values, and creates new features for further analysis.

In [ ]:
import pandas as pd
import numpy as np

## Load and Inspect the Dataset

In [ ]:
df = pd.read_csv("dataset.csv", sep=';')
print("(Nb rows, Nb columns): ", df.shape)
df.info()

In [ ]:
df.head()

In [ ]:
print(df.columns)

In [ ]:
nb_duplicates = df.duplicated().sum()

if nb_duplicates > 0:
    df = df.drop_duplicates()

print("Nb duplicates: ", nb_duplicates)
print("(Nb rows, Nb columns): ", df.shape)

## Data Cleaning

In [ ]:
df['Date'] = df['Date'].str.replace('/', '-')
df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m', errors="coerce")

numeric_cols = ['Average journey time',
                'Number of scheduled trains',
                'Number of cancelled trains',
                'Number of trains delayed at departure',
                'Average delay of late trains at departure',
                'Average delay of all trains at departure',
                'Number of trains delayed at arrival',
                'Average delay of late trains at arrival',
                'Average delay of all trains at arrival',
                'Number of trains delayed > 15min',
                'Average delay of trains > 15min (if competing with flights)',
                'Number of trains delayed > 30min',
                'Number of trains delayed > 60min',
                'Pct delay due to external causes',
                'Pct delay due to infrastructure',
                'Pct delay due to traffic management',
                'Pct delay due to rolling stock',
                'Pct delay due to station management and equipment reuse',
                'Pct delay due to passenger handling (crowding, disabled persons, connections)']

for col in numeric_cols:
    df[col] = df[col].astype(str).str.replace(',', '.')
    df[col] = pd.to_numeric(df[col], errors='coerce')

df[numeric_cols] = df[numeric_cols].fillna(0)

df.info()

In [ ]:
print(df.isnull().sum())

The dataset is checked for missing values so that incomplete rows or invalid values can be handled before feature creation.

In [ ]:
df = df.dropna(subset=["date", "scheduled_time", "actual_time", "delay_minutes"])
print("(Nb rows, Nb columns): ", df.shape)

Rows with missing values in the key columns are removed because these fields are necessary for reliable analysis and later feature engineering.

## Feature Engineering

New features are created from the original columns to make the dataset easier to analyze. In this section, temporal features such as year, month, quarter, season, and scheduled hour are added, as well as route, peak-hour, and delay-category features.

In [ ]:
df['Year'] = df['date'].dt.year
df['Month'] = df['date'].dt.month

In [ ]:
df["Month name"] = df["date"].dt.month_name()
df["Quarter"] = df["date"].dt.quarter

In [ ]:
df["Season"] = df["Month"].map({
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Autumn", 10: "Autumn", 11: "Autumn"
})

In [ ]:
df["Route"] = df["departure_station"] + " -> " + df["arrival_station"]

In [ ]:
df["scheduled_hour"] = df["scheduled_time"].apply(
    lambda x: x.hour if pd.notnull(x) else np.nan
)

In [ ]:
df["is_peak_hour"] = df["scheduled_hour"].apply(
    lambda x: 1 if x in [7, 8, 9, 17, 18, 19] else 0
)

In [ ]:
df["delay_category"] = pd.cut(
    df["delay_minutes"],
    bins=[-1, 5, 15, 30, float("inf")],
    labels=["On time / slight", "Short delay", "Medium delay", "Long delay"]
)

In [ ]:
df[["Year", "Month", "scheduled_hour", "is_peak_hour", "delay_category", "Route"]].head()

In [ ]:
df.to_csv('cleaned_dataset.csv', index=False)

## Delay Minutes Analysis

In [ ]:
df["delay_minutes"].describe()

In [ ]:
print("Mean delay:", df["delay_minutes"].mean())
print("Median delay:", df["delay_minutes"].median())
print("Minimum delay:", df["delay_minutes"].min())
print("Maximum delay:", df["delay_minutes"].max())

In [ ]:
df["delay_minutes"].plot(kind="hist", bins=10, title="Distribution of delay minutes")

The average delay is 18 minutes and the median delay is 15 minutes. This suggests that most delays are moderate, but some larger delays may increase the average. The delay values range from 5 to 35 minutes, showing that the dataset includes both shorter and longer delays.

In [ ]:
df["delay_minutes"].plot(kind="box", title="Boxplot of delay minutes")

In [ ]:
df.sort_values("delay_minutes", ascending=False).head()

## Average Delay by Departure Station

In [ ]:
departure_delay = df.groupby("departure_station")["delay_minutes"].mean().sort_values(ascending=False)
departure_delay

In [ ]:
departure_delay.plot(kind="bar", title="Average Delay by Departure Station")

This analysis compares the average delay across departure stations. Grouping the data by `departure_station` helps identify whether some stations tend to have higher delays than others. The bar chart makes it easier to compare stations and observe which ones have the highest average delay.